## 3. `calc_damping()` → damp_score_mult, damp_score_offset (with `last_supper()`) ⬜ 🟠 ❓ (VIII.10, VIII.11)

**last_supper** = the last 3 cooked meals, ignored if cooked more than 4 days ago. Their ingredients come from `recipe_ingredients`, the ingredient attributes (`veg_type`, `side_dish`) from `ingredients.yaml`. Every ingredient there can also be a recipe's `main_ingredient`.

Build **one** merged list from all last_supper meals (once, before looping the candidates):

| List | Content |
|---|---|
| `ls_origins` | all origins |
| `ls_methods` | all cooking methods |
| `ls_specs` | only sweet, soup, casserole (no other specs) |
| `ls_sides` | all ingredients with `side_dish: true` |
| `ls_mains` | the `main_ingredient` of each meal |
| `ls_meats` | all ingredients with `veg_type == meat` (each one counts for `len`) |

**Rules per candidate** – start with `damp_score_mult = 1`, `damp_score_offset = 0`:

| Condition | Effect                             |
|---|------------------------------------|
| season overlap (current season, "all" also) | offset + 3                         |
| candidate origin in `ls_origins` | offset − 1                         |
| candidate cooking method in `ls_methods` | nothing                            |
| candidate sweet, soup or casserole in `ls_specs` | mult · (1 − 0.8)                   |
| candidate side_dish in `ls_sides` | offset − 3                         |
| candidate is meat | mult · (1 − 0.1 · len(`ls_meats`)) |
| candidate meat in `ls_meats` | offset − 3                         |
| candidate `main_ingredient` in `ls_mains`, and that ingredient is meat or `side_dish: true` | mult · (1 − 0.1)                   |
| candidate `main_ingredient` in `ls_mains`, otherwise | offset − 3                         |

→ every rule only checks "in list: yes / no" → fires at most once per candidate. <br>
— note: (1−x)ⁿ ≥ 1−nx
❓ Note: a negative offset might be too strong (VIII.10).

In [ ]:
import pandas as pd
import numpy as np

today = pd.Timestamp.today().normalize()

cooked_meals = pd.DataFrame({
    "recipe_id": [1, 2, 3, 4],
    "cooked_at": today - pd.to_timedelta([1, 2, 6, 3], unit="D"),
})
# like ingredients.yaml: attributes belong to the ingredient, not to the recipe
ingredients = pd.DataFrame({
    "ingredient": ["pasta", "rice", "potato", "beef", "pork", "tofu", "apple", "salmon", "lentils", "leek"],
    "veg_type":   ["vegan", "vegan", "vegan", "meat", "meat", "vegan", "vegan", "fish", "vegan", "vegan"],
    "side_dish":  [True, True, True, False, False, False, False, False, False, False],
})
recipes = pd.DataFrame({
    "id":              [1, 2, 3, 4, 5, 6, 7, 8, 9],
    "name":            ["beef lasagne", "tofu wok", "lentil soup", "potato pork pan", "apple pie",
                        "salmon pasta", "beef burrito", "tofu curry", "potato leek soup"],
    "origin":          ["italian", "asian", "german", "italian", "asian", "french", "mexican", "asian", "german"],
    "cooking_method":  ["bake", "fry", "boil", "boil", "fry", "bake", "grill", "fry", "boil"],
    "spec":            ["casserole", None, "soup", None, "sweet", None, None, None, "soup"],
    "season":          [["all"], ["autumn"], ["winter"], ["summer"], ["autumn", "winter"], ["spring"], ["all"], ["all"], ["all"]],
    "main_ingredient": ["beef", "tofu", "lentils", "potato", "apple", "salmon", "beef", "tofu", "potato"],
})
recipe_ingredients = pd.DataFrame({
    "recipe_id":  [1, 1, 2, 2, 3, 4, 4, 5, 6, 6, 7, 7, 8, 8, 9, 9],
    "ingredient": ["pasta", "beef", "rice", "tofu", "lentils", "potato", "pork", "apple",
                   "pasta", "salmon", "rice", "beef", "tofu", "rice", "potato", "leek"],
})

In [ ]:
SPECS = {"sweet", "soup", "casserole"}
ING = ingredients.set_index("ingredient")

def current_season(d):
    return {12: "winter", 1: "winter", 2: "winter",
            3: "spring", 4: "spring", 5: "spring",
            6: "summer", 7: "summer", 8: "summer"}.get(d.month, "autumn")

def ingr_of(recipe_ids):
    """recipe_ingredients joined with the ingredient attributes."""
    return (recipe_ingredients[recipe_ingredients["recipe_id"].isin(recipe_ids)]
            .merge(ingredients, on="ingredient", how="left"))

recent = cooked_meals[cooked_meals["cooked_at"] >= today - pd.Timedelta(days=4)]
last_supper = recent.sort_values("cooked_at", ascending=False)["recipe_id"].head(3).to_numpy()

ls_recipes = recipes[recipes["id"].isin(last_supper)]
ls_ingr    = ingr_of(last_supper)

ls_origins = set(ls_recipes["origin"].dropna())
ls_methods = set(ls_recipes["cooking_method"].dropna())
ls_specs   = set(ls_recipes["spec"].dropna()) & SPECS
ls_sides   = set(ls_ingr.loc[ls_ingr["side_dish"] == True, "ingredient"])
ls_mains   = set(ls_recipes["main_ingredient"].dropna())
ls_meats   = list(ls_ingr.loc[ls_ingr["veg_type"] == "meat", "ingredient"])   # Liste: len() zählt jedes Fleisch

print(last_supper, ls_origins, ls_methods, ls_specs, ls_sides, ls_mains, ls_meats, sep="\n")

In [ ]:
def damp_score(rec, ingr, season):
    mult, offset = 1.0, 0.0
    sides = set(ingr.loc[ingr["side_dish"] == True, "ingredient"])
    meats = set(ingr.loc[ingr["veg_type"] == "meat", "ingredient"])
    main  = rec["main_ingredient"]

    if {season, "all"} & set(rec["season"]):   offset += 3
    if rec["origin"] in ls_origins:            offset -= 1
    if rec["cooking_method"] in ls_methods:    pass
    if rec["spec"] in ls_specs:                mult *= 1 - 0.8
    if sides & ls_sides:                       offset -= 3
    if meats:                                  mult *= max(0, 1 - 0.1 * len(ls_meats))
    if meats & set(ls_meats):                  offset -= 3

    if pd.notna(main) and main in ls_mains:
        # main_ingredient is an ingredient from ingredients.yaml -> use its own attributes
        if ING.loc[main, "veg_type"] == "meat" or ING.loc[main, "side_dish"]:
            mult *= 1 - 0.1
        else:
            offset -= 3
    return mult, offset

In [ ]:
season = current_season(today)
candidates = recipes[~recipes["id"].isin(last_supper)]

rows = []
for _, rec in candidates.iterrows():
    ingr = ingr_of([rec["id"]])
    damp_score_mult, damp_score_offset = damp_score(rec, ingr, season)
    rows.append({"id": rec["id"], "name": rec["name"],
                 "damp_score_mult": round(damp_score_mult, 3),
                 "damp_score_offset": damp_score_offset})

scores = pd.DataFrame(rows)
scores

## Sanity Checks

Expected behavior. 🟠 = not tested yet; "old" = value from the old test script, re-check here.

| Situation | Expected | Status |
|---|---|---|
| same meat + side + main as yesterday | strongly damped, but still positive | 🟠 old: 3.72 at base 12 |
| other meat type than last_supper | only ×(1 − 0.1·n), no −3 | 🟠 |
| same main, neither meat nor side (tofu curry) | −3 | 🟠 |
| same main, and it is a side_dish ingredient (potato leek soup) | ×0.9 | 🟠 |
| soup after soup | strongly damped | 🟠 old: ×0.2 |
| nothing in common with last_supper | unchanged (+ season) | 🟠 |
| every damping rule | fires max. once | 🟠 |

## Edge Cases – defaults to choose

❓ = not decided yet, pick a default (first option = current / suggested).

| Case | Default now | Options to choose |
|---|---|---|
| nothing cooked in the last 4 days (empty last_supper) | `mult = 1`, offset only the season bonus | – |
| fewer than 3 meals in the last 4 days | use the ones there are | – |
| field missing (`origin`, `cooking_method`, `spec`, `main_ingredient`) | rule doesn't fire | – |
| `season` missing | counts as `all` → +3 | – |
| `main_ingredient = "not specific"` | treated as missing → no main rule | – |
| `main_ingredient` not in `ingredients.yaml` | ❓ currently `KeyError` in `ING.loc` | neither meat nor side → −3 · skip the main rule |
| `main_ingredient` not in the recipe's own ingredients | uses the attributes from `ingredients.yaml` anyway | – |
| same meat in several last_supper meals | counts several times in `len(ls_meats)` | ❓ keep · count each meat once (`set`) |
| `len(ls_meats) ≥ 10` | meat factor clamped to 0 | – (can't happen with 3 meals unless a recipe has many meats) |
| candidate has several meats | meat factor applied once | – |
| several origins (list) | any overlap → −1 once | – |
| sweet dish via recipe flag `sweet` (not `spec`) | counts as spec `sweet` | – |
| variant / original of a last_supper recipe | removed by the hard filter before damping | – |